In [1]:
import pandas as pd
import xarray as xr
import numpy as np
import dask.array as da
import dask.dataframe as dd
import cftime
import os,warnings
warnings.filterwarnings('ignore') # Turns off warnings

#def fast_csv_to_netcdf(input_csv, output_nc):
    # Read CSV using dask
   # df = dd.read_csv(input_csv, 
                     dtype={
                         'X': 'float32', 
                         'Y': 'float32', 
                         'Value': 'float32', 
                         'Month': 'int32', 
                         'Year': 'int32'
                     })
    
    # Get unique coordinates
    x_coords = sorted(df['X'].unique().compute())
    y_coords = sorted(df['Y'].unique().compute())
    
    # Get time coordinates more efficiently
    time_df = df[['Year', 'Month']].drop_duplicates().compute()
    time_coords = sorted([cftime.Datetime360Day(y, m, 1) 
                         for y, m in zip(time_df['Year'], time_df['Month'])])
    
    # Compute the DataFrame
    df_computed = df.compute()
    
    # Handle duplicates by averaging values for the same coordinates
    df_grouped = df_computed.groupby(['Year', 'Month', 'Y', 'X'])['Value'].mean()
    
    # Create a 3D numpy array filled with NaN
    shape = (len(time_coords), len(y_coords), len(x_coords))
    data_array = np.full(shape, np.nan, dtype='float32')
    
    # Create lookup dictionaries for faster indexing
    time_lookup = {(d.year, d.month): i for i, d in enumerate(time_coords)}
    y_lookup = {y: i for i, y in enumerate(y_coords)}
    x_lookup = {x: i for i, x in enumerate(x_coords)}
    
    # Fill the array with values
    for (year, month, y, x), value in df_grouped.items():
        t_idx = time_lookup.get((year, month))
        y_idx = y_lookup.get(y)
        x_idx = x_lookup.get(x)
        if all(idx is not None for idx in (t_idx, y_idx, x_idx)):
            data_array[t_idx, y_idx, x_idx] = value
    
    # Create xarray Dataset directly
    ds = xr.Dataset(
        data_vars={
            'snow_water_equivalent': (['time', 'Y', 'X'], data_array)
        },
        coords={
            'time': time_coords,
            'Y': y_coords,
            'X': x_coords
        }
    )
    
    # Add metadata
    ds.snow_water_equivalent.attrs = {
        'units': 'mm',
        'long_name': 'Snow Water Equivalent'
    }
    
    # Save with efficient encoding
    encoding = {
        'time': {
            'calendar': '360_day',
            'units': 'days since 1900-01-01'
        },
        'snow_water_equivalent': {
            'zlib': True,
            'complevel': 5,
            'chunksizes': (1, len(y_coords), len(x_coords))
        }
    }
    
    ds.to_netcdf(output_nc, encoding=encoding)
    return ds

# Usage
if __name__ == '__main__':
    input_csv = 'combined_snow_water.csv'
    output_nc = 'snow_water_equivalent.nc'
    ds = fast_csv_to_netcdf(input_csv, output_nc)

/home/aelyoussoufi/miniconda3/envs/latest/lib/python3.10/site-packages/dask/dataframe/__init__.py:49: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [5]:
xr.open_mfdataset(output_nc)

<xarray.Dataset> Size: 5GB
Dimensions:                (time: 52, Y: 3351, X: 6935)
Coordinates:
  * time                   (time) object 416B 2003-12-01 00:00:00 ... 2020-12...
  * Y                      (Y) float64 27kB 24.95 24.96 24.97 ... 52.86 52.87
  * X                      (X) float64 55kB -124.7 -124.7 ... -66.95 -66.95
Data variables:
    snow_water_equivalent  (time, Y, X) float32 5GB dask.array<chunksize=(1, 3351, 6935), meta=np.ndarray>